# 八幡浜みかん出荷量予測パイプライン — 完全解説

衛星画像 + 農地ポリゴン + 政府統計を組み合わせて、愛媛県八幡浜市の温州みかん年間出荷量（トン）を予測する機械学習パイプラインの全体説明。

**予測目標**: 衛星データから年間出荷量を推定し MAPE < 20% を達成する

---

## パイプライン全体図

```
┌─────────────────────────────────────────────────────────────────────┐
│                        データソース                                  │
│                                                                     │
│  [A] MAFF 筆ポリゴン   [B] e-Stat API          [C] Google Earth Engine │
│  2024_382043.geojson   出荷量統計               Sentinel-2/S1/Landsat  │
│  37,716ポリゴン        1993-2006直接測定         衛星リモートセンシング   │
│  land_type=200/100     2007-2016疑似ラベル                            │
└──────────┬──────────────────┬──────────────────────┬────────────────┘
           │                  │                      │
           ▼                  │                      ▼
  ┌──────────────────┐        │        ┌─────────────────────────────┐
  │  Phase 1: 前処理  │        │        │ Phase 2: GEE特徴量生成       │
  │  fude.py         │        │        │ gee_features.py             │
  │  ・地目フィルタ    │        │        │ Sentinel-2: 5指数×12ヶ月=60  │
  │  ・面積フィルタ    │        │        │ Sentinel-1: 3バンド×12=36    │
  │  ・地形フラグ付与  │        │        │ SRTM地形:  elevation/slope/  │
  │  19,258ポリゴン  │        │        │            aspect = 3        │
  └────────┬─────────┘        │        │ → 99バンド / ポリゴン        │
           │                  │        └──────────────┬──────────────┘
           │                  │                       │
           ▼                  │                       ▼
  ┌──────────────────┐        │        ┌─────────────────────────────┐
  │  Stage-1 分類    │◄───────┘        │ Phase 4: Landsat年次特徴量   │
  │  Random Forest   │                 │ gee_landsat.py              │
  │  畑 vs 田 判別   │                 │ NDVI夏/冬, NDMI, 面積推定    │
  │  OOB: 0.9934    │                 │ 1994-2014年 → 8特徴量        │
  │  面積推定: 2698ha│                 └──────────────┬──────────────┘
  └────────┬─────────┘                               │
           │ stage1_area_ha                          │
           └──────────────────────┬──────────────────┘
                                  │
                                  ▼
                     ┌─────────────────────────┐
                     │  Stage-2 回帰           │
                     │  XGBoost + LOOCV        │
                     │  入力: Landsat 8特徴量   │
                     │       + Stage-1面積      │
                     │  目標: e-Stat出荷量       │
                     │  MAPE(直接測定): 12.5%   │
                     └─────────────────────────┘
```

In [ ]:
import sys
from pathlib import Path

BASE = Path('../')
sys.path.insert(0, str(BASE / 'src'))
sys.path.insert(0, str(BASE.parent / 'src'))  # gee_utils

import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.dpi'] = 120
print('準備完了')

---
## データソース A: MAFF 筆ポリゴン

### 何のデータか
農林水産省が整備した **農地の区画情報（筆ポリゴン）**。
全国の農地を1筆ごとにポリゴンで表し、地目コードを付与している。

- **提供元**: 農林水産省 農業農村整備情報総合センター  
- **ファイル**: `2024_382043.geojson`（市区町村コード 382043 = 八幡浜市）  
- **取得方法**: [open.fude.maff.go.jp](https://open.fude.maff.go.jp) でユーザー登録後にダウンロード

### 主要フィールド
| フィールド | 内容 |
|-----------|------|
| `polygon_uuid` | 全国一意のポリゴンID |
| `land_type` | **地目コード**: 100=田、200=畑 |
| `issue_year` | 筆ポリゴン作成年 |
| `point_lng/lat` | 代表点座標 |

### なぜみかん検出に使えるか
みかんは果樹（畑作）なので **land_type=200（畑）** に分類される。  
ただし筆ポリゴンには **作物種別コードは存在しない**ため、「畑=みかん候補」という仮定を使う。  
田（land_type=100）はネガティブサンプル（確実にみかんではない）として利用。

In [ ]:
# 生の筆ポリゴンデータ
raw_fude = gpd.read_file(BASE / 'data/raw/fude/2024_382043.geojson')
print(f'総ポリゴン数: {len(raw_fude):,}')
print(f'カラム: {list(raw_fude.columns)}')
print(f'\n地目コード分布:')
print(raw_fude['land_type'].value_counts().rename({100: '100=田', 200: '200=畑'}))

In [ ]:
# フィルタ後（fude.py処理済み）
fude = gpd.read_file(BASE / 'data/processed/fude_filtered.geojson')
print(f'フィルタ後ポリゴン数: {len(fude):,}')
print(f'  畑 (label=1): {(fude["label"]==1).sum():,}')
print(f'  田 (label=0): {(fude["label"]==0).sum():,}')
print(f'  南向き地形OK: {fude["terrain_ok"].sum():,} ({fude["terrain_ok"].mean()*100:.1f}%)')
print(f'  面積統計 (m²):')
print(fude['area_m2'].describe().apply(lambda x: f'{x:,.0f}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ラベル分布
ax = axes[0]
counts = fude['label'].value_counts().sort_index()
bars = ax.bar(['Paddy (label=0)', 'Upland field (label=1)'], counts.values,
              color=['steelblue', 'darkorange'])
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, f'{v:,}',
            ha='center', fontsize=10)
ax.set_title('Label distribution (after filtering)')
ax.set_ylabel('# polygons')

# 面積分布
ax = axes[1]
ax.hist(fude['area_m2'].clip(0, 5000), bins=50, color='darkorange', edgecolor='white')
ax.axvline(500, color='red', linestyle='--', label='500 m² threshold')
ax.set_title('Polygon area distribution')
ax.set_xlabel('Area (m²)')
ax.set_ylabel('Count')
ax.legend()

plt.suptitle('MAFF Fude Polygons — Yawatahama (382043)', fontsize=13)
plt.tight_layout()
plt.show()

### 前処理フロー (fude.py)

```
37,716 ポリゴン (生データ)
   │
   ├─ land_type=200 → label=1 (みかん候補)  37,237件
   ├─ land_type=100 → label=0 (ネガティブ)     479件
   │
   ▼ 面積フィルタ (≥500m²)
19,258 ポリゴン
   │
   ▼ 地形フラグ (GEE SRTM)
   ├─ terrain_ok=1: 南向き (aspect 135-225°) かつ傾斜 10-40°  → 10,261件
   └─ terrain_ok=0: それ以外                                   →  8,997件
```

**地形フラグの取得方法**: GEEの `ee.Terrain.slope/aspect(SRTM)` で全ポリゴンを500件ずつバッチ処理し、`reduceRegions()` で各ポリゴンの地形を判定。

---
## データソース B: e-Stat 政府統計API — 出荷量データ

### 何のデータか
総務省統計局が提供する **政府統計の総合窓口 (e-Stat)** のAPI。  
農林水産省の果樹統計から八幡浜市のみかん出荷量を取得する。

### なぜ3つの統計IDが必要か

| 期間 | statsId | 統計名 | 取得内容 | is_proxy |
|------|---------|--------|---------|----------|
| 1993-2005 | `0003274240` | 果樹累年統計 | 八幡浜市 (cdArea=38204), 出荷量 (cdCat01=120) | `False` |
| 2006 | `0003022267` | 市町村別統計 | 八幡浜市 (cat01=346), 出荷量 (cat02=004) | `False` |
| 2007-2016 | `0003313868` | 愛媛県計 | 愛媛県 (cdArea=38000), 出荷量 (cdCat01=170) | **`True`** |

2007年以降は市町村別データが公開されていないため、**愛媛県計 × 推計シェア 31%** で代用。

### 31%（推計シェア）の根拠
2005-2006年の実績から計算した八幡浜市の県内シェア:
```
2005年: 50,800t (八幡浜) ÷ 164,900t (愛媛県計) ≈ 30.8%
2006年: 37,000t (八幡浜) ÷ 124,100t (愛媛県計) ≈ 29.8%
平均: ≈ 30.3%  → config: yawatahama_share_proxy = 0.31
```
**これは推計値であり実測値ではない。** 疑似ラベル年 (2007-2016) のデータは `is_proxy=True` でマークし、モデル評価では直接測定年 (1993-2006) のMAPEを主要指標とする。

In [ ]:
shipment = pd.read_csv(BASE / 'data/raw/stats/yawatahama_shipment.csv')
print(f'総行数: {len(shipment)}')
print(f'直接測定年 (is_proxy=False): {(~shipment["is_proxy"]).sum()}件 '
      f'({shipment[~shipment["is_proxy"]]["year"].min()}-{shipment[~shipment["is_proxy"]]["year"].max()})')
print(f'疑似ラベル年 (is_proxy=True) : {shipment["is_proxy"].sum()}件 '
      f'({shipment[shipment["is_proxy"]]["year"].min()}-{shipment[shipment["is_proxy"]]["year"].max()})')
print()
print(shipment.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

direct = shipment[~shipment['is_proxy']]
proxy  = shipment[shipment['is_proxy']]

ax.bar(direct['year'], direct['shipment_t'] / 1000, color='steelblue', label='Direct measurement (e-Stat 38204)')
ax.bar(proxy['year'],  proxy['shipment_t']  / 1000, color='lightgray', label='Proxy: Pref total × 31%', edgecolor='gray')

ax.axvspan(2006.5, 2016.5, alpha=0.06, color='gray')
ax.text(2011, 54, 'Proxy years\n(estimated)', ha='center', color='gray', fontsize=9)

ax.set_title('Yawatahama Mikan Shipment — e-Stat Data')
ax.set_xlabel('Year')
ax.set_ylabel('Shipment (1000 t)')
ax.set_ylim(0, 60)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### e-Stat APIリクエストの構造

```python
# 1993-2005年: 市区町村別・果樹累年統計
GET https://api.e-stat.go.jp/rest/3.0/app/json/getStatsData
  ?statsDataId=0003274240
  &cdArea=38204       # 八幡浜市
  &cdCat01=120        # みかん出荷量
  &appId=<API_KEY>

# 2006年: 市区町村別統計（別IDが必要）
GET .../getStatsData
  ?statsDataId=0003022267
  &cdCat01=346        # 八幡浜市コード（この統計ではcat01が市区町村）
  &cdCat02=004        # 出荷量

# 2007-2016年: 愛媛県計（市区町村別データなし）
GET .../getStatsData
  ?statsDataId=0003313868
  &cdCat01=170        # 温州みかん出荷量
  &cdCat02=0110
  &cdArea=38000       # 愛媛県
# → 取得値 × 0.31 = 八幡浜推計値
```

---
## データソース C-1: Google Earth Engine — Sentinel-2/S1 特徴量

### 何のデータか
- **Sentinel-2**: ESA/Copernicus の光学衛星。10m解像度、5日周期。2015年以降
- **Sentinel-1**: ESA の SAR (合成開口レーダー) 衛星。雲の影響なし。10m解像度
- **SRTM**: NASAのシャトルレーダー地形ミッション。30m解像度の標高データ

### なぜ月次コンポジットを使うか
単一画像は雲や大気の影響を受けやすい。各月の複数画像のメディアン合成（Cloud Score+でマスク後）により安定した値を得る。

### 99バンドの構成

| カテゴリ | バンド | 月数 | 計 | 意味 |
|---------|--------|------|----|------|
| Sentinel-2 | NDVI, NDre, NDMI, NDWI, GNDVI | 12 | 60 | 植生・水分・葉緑素 |
| Sentinel-1 | VH, VV, VH/VV比 | 12 | 36 | 構造・水分（SAR）|
| SRTM地形 | elevation, slope, aspect | — | 3 | 高度・傾斜・向き |
| **合計** | | | **99** | |

### みかん検出に重要な指数
| 指数 | 算式 | みかんへの意味 |
|-----|------|---------------|
| NDVI | (NIR-Red)/(NIR+Red) | 植生量。常緑のみかんは冬でも高い |
| NDre | (NIR-RedEdge)/(NIR+RedEdge) | 葉緑素量。着果期に変化 |
| NDMI | (NIR-SWIR1)/(NIR+SWIR1) | 葉の水分含量 |
| GNDVI | (NIR-Green)/(NIR+Green) | 葉面積指数と相関 |

In [ ]:
feat_s2 = gpd.read_file(BASE / 'data/processed/features_s2.geojson')
print(f'ポリゴン数: {len(feat_s2):,}')
print(f'総カラム数: {len(feat_s2.columns)} (99バンド + polygon_id, label, terrain_ok, area_m2, geometry)')

# バンド名一覧
feature_cols = [c for c in feat_s2.columns if c not in {'polygon_id','label','terrain_ok','area_m2','geometry'}]
s2_cols = [c for c in feature_cols if any(c.startswith(p) for p in ['NDVI','NDre','NDMI','NDWI','GNDVI'])]
s1_cols = [c for c in feature_cols if any(c.startswith(p) for p in ['VH','VV'])]
ter_cols = [c for c in feature_cols if c in ['elevation','slope','aspect']]
print(f'\nS2バンド ({len(s2_cols)}): {s2_cols[:6]} ... {s2_cols[-3:]}')
print(f'S1バンド ({len(s1_cols)}): {s1_cols[:6]} ... {s1_cols[-3:]}')
print(f'地形バンド ({len(ter_cols)}): {ter_cols}')

In [ ]:
# 月別NDVI: みかん（畑）vs 田の比較
months = list(range(1, 13))
ndvi_cols = [f'NDVI_{m:02d}' for m in months]

fig, ax = plt.subplots(figsize=(10, 4))

for label, color, name in [(1, 'darkorange', 'Upland field / mikan candidate (label=1)'),
                            (0, 'steelblue',  'Paddy field (label=0)')]:
    subset = feat_s2[feat_s2['label'] == label][ndvi_cols].dropna()
    mean_ndvi = subset.mean()
    std_ndvi  = subset.std()
    ax.plot(months, mean_ndvi.values, '-o', color=color, label=name, linewidth=2)
    ax.fill_between(months, mean_ndvi - std_ndvi, mean_ndvi + std_ndvi,
                    alpha=0.15, color=color)

ax.set_xticks(months)
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_xlabel('Month (2022)')
ax.set_ylabel('NDVI (mean ± 1σ)')
ax.set_title('Monthly NDVI by land type — Sentinel-2 2022')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('※ みかん（常緑）は冬でもNDVI高め。田は夏に高く冬に低い（稲作の季節変化）')

In [ ]:
# GEEでの取得方法（コード概要）
overview = '''
## Sentinel-2 月次コンポジット生成 (gee_features.py)

1. S2コレクションとCloud Score+コレクションをJoinで結合
   s2_col.filterDate(start, end).filterBounds(aoi)
   + ee.Join.saveFirst("cs_image")

2. cs > 0.65 のピクセルのみ使用（雲マスク）
   masked = img.updateMask(cs_img.gt(0.65)).multiply(0.0001)

3. 月内の全画像をメディアン合成 → .median()

4. 5つの指数を計算して名前付き
   NDVI_01, NDre_01, NDMI_01, NDWI_01, GNDVI_01  (1月)
   ...以下12月まで...
   NDVI_12, NDre_12, NDMI_12, NDWI_12, GNDVI_12  (12月)

5. 全19,258ポリゴンにreduceRegions(reducer=mean, scale=10m)
   → 500件ずつバッチ処理 (GEE payload制限対策)
   → geemap.ee_to_gdf() でGeoDataFrameに変換 (Drive不要)
'''
print(overview)

---
## データソース C-2: Google Earth Engine — Landsat 年次特徴量

### なぜLandsatが必要か
**Sentinel-2は2015年以降しかない**が、e-Statの市区町村別出荷量統計は **2006年が最後**。
両者が重なる期間がないため、Sentinel-2は空間分類（Stage-1）に使い、  
回帰（Stage-2）は **Landsat（1993年〜）** の長期データを使う2層構造にした。

```
時系列
────────────────────────────────────────────────────────────────
1993   1995   1997   1999   2001   2003   2005   2007  ... 2015 ... 2023
│←──────── e-Stat 市区町村データ (直接測定) ──────────────────►│  (消滅)
│←──────── Landsat 4/5/7/8 (1993〜) ─────────────────────────────────►│
                                                              │←Sentinel-2─►│
         
Stage-2入力                                         Stage-1入力
(Landsat特徴量 + e-Stat出荷量)                       (S2特徴量 + 筆ポリゴンラベル)
```

### Landsat年次特徴量の内容 (gee_landsat.py)

| 特徴量 | 算出方法 | 意味 |
|--------|---------|------|
| `ndvi_summer_mean` | 6-8月NDVI メディアン合成の平均 | 夏の植生量 |
| `ndvi_summer_std` | 同標準偏差 | 空間的ばらつき（みかん園地の均一性） |
| `ndvi_winter` | 12-2月NDVI 中央値 | 冬の常緑樹量（みかんは高い） |
| `ndmi_summer` | 夏のNDMI平均 | 夏の水分状態 |
| `area_sat_ha` | `make_mikan_mask()` でみかん候補と判定されたピクセル面積 | 衛星推定みかん面積 |
| `area_delta_ha` | 前年比の面積変化 | 年間変動 |
| `n_images` | 使用した画像枚数 | データ品質の指標 |

In [ ]:
landsat = pd.read_csv(BASE / 'data/processed/features_landsat.csv')
print(f'行数: {len(landsat)}年分 ({landsat["year"].min()}-{landsat["year"].max()})')
print(f'カラム: {list(landsat.columns)}')
print()
print(landsat.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

plots = [
    ('ndvi_summer_mean', 'NDVI Summer Mean', 'darkorange'),
    ('ndvi_winter',      'NDVI Winter',       'steelblue'),
    ('ndmi_summer',      'NDMI Summer',       'seagreen'),
    ('area_sat_ha',      'Sat-estimated Mikan Area (ha)', 'tomato'),
]

for ax, (col, title, color) in zip(axes.flat, plots):
    ax.plot(landsat['year'], landsat[col], '-o', color=color, linewidth=2, markersize=5)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.grid(alpha=0.3)

plt.suptitle('Landsat Annual Features — Yawatahama AOI', fontsize=13)
plt.tight_layout()
plt.show()
print('注: Landsatは雲カバー率によっては特定の年のデータが少ない (n_imagesが小さい年は不安定)')

---
## Stage-1: みかん園地検出（Random Forest）

### 学習の仕組み

```
入力: features_s2.geojson
  ・99バンド (Sentinel-2/S1/地形)
  ・19,258ポリゴン

ラベル:
  ・label=1 (畑, land_type=200) = 18,800件  ← みかん候補
  ・label=0 (田, land_type=100) =    458件  ← 確実な非みかん

空間分割 (リーケージ防止):
  ・EPSG:6677座標系で緯度順にソート
  ・後ろ20%（北部エリア）をテストセット

モデル:
  RandomForestClassifier(
    n_estimators=300,
    oob_score=True,
    class_weight='balanced',   # 79:1の不均衡対策
    n_jobs=-1
  )
```

### 結果の解釈
OOB Score 0.9934は「畑と田を区別できている」指標。  
ただし作物種別は識別できないため、**全畑≈みかん という仮定の上での結果**。  
特徴量重要度が本来の価値で、「どの波長・季節・地形がみかんと他作物の違いに寄与するか」を示す。

In [ ]:
with open(BASE / 'outputs/models/stage1_rf.pkl', 'rb') as f:
    stage1 = pickle.load(f)

rf = stage1['model']
feature_cols = stage1['feature_cols']

print(f'OOB Score: {rf.oob_score_:.4f}')
print(f'特徴量数: {len(feature_cols)}')
print(f'木の本数: {rf.n_estimators}')

# 特徴量重要度
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
top20 = importances.head(20)
print(f'\n上位5特徴量:')
for name, imp in top20.head(5).items():
    month = name.split('_')[-1] if '_' in name else '-'
    print(f'  {name}: {imp:.4f}  (月: {month})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 特徴量重要度
ax = axes[0]
top20[::-1].plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Stage-1 RF Feature Importance TOP20')
ax.set_xlabel('Importance')
ax.tick_params(axis='y', labelsize=8)

# センサー別・月別重要度ヒートマップ
ax = axes[1]
index_names = ['NDVI','NDre','NDMI','NDWI','GNDVI']
matrix = np.zeros((len(index_names), 12))
for i, idx_name in enumerate(index_names):
    for m in range(1, 13):
        col = f'{idx_name}_{m:02d}'
        if col in importances.index:
            matrix[i, m-1] = importances[col]

im = ax.imshow(matrix, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(len(index_names)))
ax.set_yticklabels(index_names)
ax.set_xticks(range(12))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.set_title('S2 Feature Importance by Index × Month')
ax.set_xlabel('Month')
plt.colorbar(im, ax=ax, label='Importance')

plt.tight_layout()
plt.show()
print('→ 3月（冬明け〜春先）が最も重要: 常緑みかんと休耕田・落葉樹の差が最大になる時期')

In [ ]:
# Stage-1でのみかん面積推定
# モデルのfeature_colsを使う（terrain_okも含む100特徴量）
feat_s2_loaded = gpd.read_file(BASE / 'data/processed/features_s2.geojson')
_feat_cols = stage1['feature_cols']  # pkl保存済みの特徴量名リスト（terrain_ok含む）
X_all = feat_s2_loaded[_feat_cols].fillna(0).values.astype(np.float32)

probs = rf.predict_proba(X_all)[:, 1]
threshold = 0.55
mikan_mask = probs >= threshold

print(f'確率閾値: {threshold}')
print(f'みかんと判定: {mikan_mask.sum():,} / {len(mikan_mask):,} ポリゴン ({mikan_mask.mean()*100:.1f}%)')

area_ha = feat_s2_loaded.loc[mikan_mask, 'area_m2'].sum() / 10_000
print(f'Stage-1推定みかん面積: {area_ha:.1f} ha')
print()
print('※ 注意: land_type=200（畑）のほぼ全てが推定みかんと判定される')
print('  → 作物種別コードがない筆ポリゴンの限界。畑全体≈みかんの仮定が成立している地域のみ有効')

---
## Stage-2: 出荷量回帰（XGBoost + LOOCV）

### データの結合

```
features_landsat.csv    yawatahama_shipment.csv
(13行 × 8特徴量)    ×    (24行 × 3列)
        │                       │
        └───── year で inner join ──────┘
                     │
              13行 × 9特徴量
        (Landsatデータがある年のみ)
              ↓ + stage1_area_ha (定数)
              13行 × 10特徴量
```

### なぜLOOCVか
N=13（行数）は非常に少ない。通常のtrain/test分割では評価が不安定になる。  
**LOOCV（Leave-One-Out Cross-Validation）** = 1年を外してN-1年で学習→外した年を予測、全年分繰り返す。  
各年の予測が「その年を学習に使わずに」行われるため、オーバーフィット検出に適している。

### 評価の分け方
| 対象 | 年 | N | MAPE | 意味 |
|------|---|---|------|------|
| 直接測定年 | 1994,1995,1997,2000-2004,2006 | 9 | **12.5%** ← 主要指標 | e-Statで実際に測定 |
| 疑似ラベル年 | 2007,2009,2010,2014 | 4 | 20.8% ※参考値 | 県計×31%の推計値 |

In [ ]:
# データ結合の再現
feat_lc = pd.read_csv(BASE / 'data/processed/features_landsat.csv')
labels  = pd.read_csv(BASE / 'data/raw/stats/yawatahama_shipment.csv')

df = feat_lc.merge(labels, on='year', how='inner').dropna(subset=['shipment_t'])
print(f'結合後行数: {len(df)} ({df["year"].min()}-{df["year"].max()})')
print(f'  直接測定年: {(~df["is_proxy"]).sum()}件')
print(f'  疑似ラベル: {df["is_proxy"].sum()}件')
print()

# 特徴量の説明
exclude = {'year', 'shipment_t', 'is_proxy'}
feature_cols_s2 = [c for c in df.columns if c not in exclude]
print(f'特徴量: {feature_cols_s2}')

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from xgboost import XGBRegressor

exclude = {'year', 'shipment_t', 'is_proxy'}
feature_cols_xgb = [c for c in df.columns if c not in exclude]

# Stage-1面積を定数特徴量として追加
df_model = df.copy()
df_model['stage1_area_ha'] = 2697.6
feature_cols_xgb.append('stage1_area_ha')

X = df_model[feature_cols_xgb].fillna(0).values
y = df_model['shipment_t'].values

model = XGBRegressor(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.5, reg_lambda=2.0, random_state=42
)

loo = LeaveOneOut()
preds = cross_val_predict(model, X, y, cv=loo)

is_proxy = df_model['is_proxy'].values
years    = df_model['year'].values
direct   = ~is_proxy

def mape(a, p): return float(np.mean(np.abs((a - p) / a)) * 100)

print(f'MAPE (直接測定年, N={direct.sum()}): {mape(y[direct], preds[direct]):.1f}%  ← 主要指標')
print(f'MAPE (疑似ラベル年, N={(~direct).sum()}): {mape(y[~direct], preds[~direct]):.1f}%  ※参考値')
print(f'MAPE (全年, N={len(y)}): {mape(y, preds):.1f}%')

In [ ]:
# 年別詳細
result_df = pd.DataFrame({
    'year':      years,
    'actual_t':  y.astype(int),
    'pred_t':    preds.astype(int),
    'error_pct': ((preds - y) / y * 100).round(1),
    'is_proxy':  is_proxy,
})
print(result_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 時系列グラフ
ax = axes[0]
ax.plot(years, y / 1000, 'o-', color='steelblue', label='Actual (direct data)', linewidth=2, zorder=3)
ax.plot(years, preds / 1000, 's--', color='tomato', label='Predicted (LOOCV)', linewidth=2, zorder=3)

proxy_years = years[is_proxy]
if len(proxy_years):
    ax.axvspan(proxy_years.min()-0.5, proxy_years.max()+0.5, alpha=0.08, color='gray')
    ax.plot(years[is_proxy], y[is_proxy]/1000, 'o', color='gray', alpha=0.6, zorder=2)

mape_d = mape(y[direct], preds[direct])
ax.set_title(f'Yawatahama Mikan Shipment — MAPE (direct): {mape_d:.1f}%')
ax.set_xlabel('Year'); ax.set_ylabel('Shipment (1000 t)')
ax.legend(); ax.grid(alpha=0.3)

# 実測 vs 予測 散布図
ax = axes[1]
ax.scatter(y[direct]/1000, preds[direct]/1000, color='steelblue', s=80, label='Direct measurement', zorder=3)
ax.scatter(y[~direct]/1000, preds[~direct]/1000, color='lightgray', s=80, edgecolor='gray', label='Proxy label', zorder=2)

lim = (30, 55)
ax.plot(lim, lim, 'k--', alpha=0.4, label='Perfect prediction')
ax.set_xlim(*lim); ax.set_ylim(*lim)
ax.set_xlabel('Actual (1000 t)'); ax.set_ylabel('Predicted (1000 t)')
ax.set_title('Actual vs Predicted')
ax.legend(); ax.grid(alpha=0.3)

# 年ラベル
for _, row in result_df[~result_df['is_proxy']].iterrows():
    ax.annotate(str(row['year']), (row['actual_t']/1000, row['pred_t']/1000),
                textcoords='offset points', xytext=(4, 3), fontsize=7, color='steelblue')

plt.tight_layout()
plt.show()

---
## 精度評価と課題

### 現状の誤差パターン

| 誤差の種類 | 例 | 原因 |
|-----------|-----|------|
| 低値を過大推定 | 1994: +24.2% | 冷夏・病害・不作年をLandsatが捉えられない |
| 高値を過小推定 | 1997: -17.3% | 豊作年のピークを平均に引き寄せる |
| 全体に「平均回帰」 | — | N=9では極端な年を学習しきれない |

### 根本的な原因
**Landsatは植生の「平均状態」を計測するが、出荷量の年変動は以下に支配される:**
1. **隔年結果性 (alternate bearing)** — みかんは豊作年の翌年は不作になりやすい
2. **気象** — 開花期の霜、夏の干ばつ、台風被害。NDVIには出にくい
3. **価格・経済** — 農家の出荷判断に影響

### 改善可能な点

| 改善策 | 期待効果 | 実装難易度 |
|-------|---------|----------|
| `shipment_lag1`（前年出荷量）を特徴量追加 | 隔年サイクル捕捉 → MAPE大幅改善の可能性 | ★☆☆ 簡単 |
| AMeDAS気象データ（気温・降水量） | 気象要因を取り込む | ★★☆ 中程度 |
| e-Stat以外の愛媛農業統計でN増加 | N=9の問題解消 | ★★★ データ入手が課題 |
| Stage-1に作物種別情報（現地調査） | 畑≈みかん仮定の解消 | ★★★ フィールドワーク必要 |

In [ ]:
# 改善策1の試み: 前年出荷量を特徴量に追加
df_lag = df.copy()
df_lag = df_lag.sort_values('year')
df_lag['shipment_lag1'] = df_lag['shipment_t'].shift(1)
df_lag = df_lag.dropna(subset=['shipment_lag1'])

excl = {'year','shipment_t','is_proxy'}
feat_lag = [c for c in df_lag.columns if c not in excl]

X_lag = df_lag[feat_lag].fillna(0).values
y_lag = df_lag['shipment_t'].values
is_proxy_lag = df_lag['is_proxy'].values
direct_lag   = ~is_proxy_lag

preds_lag = cross_val_predict(
    XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05,
                 subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0, random_state=42),
    X_lag, y_lag, cv=LeaveOneOut()
)

print('=== 前年出荷量 (lag-1) 追加後 ===')
if direct_lag.any():
    print(f'MAPE (直接測定年, N={direct_lag.sum()}): {mape(y_lag[direct_lag], preds_lag[direct_lag]):.1f}%')
print(f'MAPE (全年, N={len(y_lag)}): {mape(y_lag, preds_lag):.1f}%')
print(f'\n(ベースライン: 直接測定年 12.5%, 全年 15.0%)')

---
## まとめ

### データフロー全体の整理

```
データ                  処理                    出力
──────────────────────────────────────────────────────
MAFF筆ポリゴン         fude.py                  fude_filtered.geojson
37,716ポリゴン     → 地目/面積/地形フィルタ →  19,258ポリゴン (label, terrain_ok)

Sentinel-2/S1/SRTM    gee_features.py           features_s2.geojson
GEE衛星データ       → 月次コンポジット×99バンド→  19,258ポリゴン × 99特徴量

                    Stage-1 RF分類             stage1_rf.pkl
features_s2        → RandomForest (畑 vs 田) →  推定みかん面積: 2,698 ha

e-Stat API          estat_shipment.py          yawatahama_shipment.csv
政府統計3種        → 直接取得+疑似ラベル生成 →  24年分の出荷量 (9直接+4proxy=使用)

Landsat 1994-2014   gee_landsat.py             features_landsat.csv
GEE長期衛星        → 年次NDVI/NDMI/面積推定 →  13年分の年次特徴量

features_landsat   Stage-2 XGBoost回帰         stage2_xgb.pkl
+ shipment_labels  → LOOCV (N=13)          →  MAPE 12.5% (直接測定9年)
```

**達成できたこと**: 衛星データのみで年間出荷量を ±12.5% の精度で推定  
**残る課題**: 隔年サイクル・気象要因の未考慮、N=9の少数データ問題